# Install all necessary libraries

In [1]:
!pip install -q openai google-generativeai # -q means silent install, logs are not displayed
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.3 MB/s eta 0:00:00


# Get API key in the file (Never share it publicly)

In [2]:
with open("Chat_bot_2.txt", "r") as f:
    Chat_bot_2 = f.read()

# Import libraries

In [3]:
import google.generativeai as genai
import streamlit as st

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


# Configure model

In [4]:
genai.configure(api_key = Chat_bot_2)

# Load Model

In [5]:
model = genai.GenerativeModel("gemini-3.5-flash")

# Generate response

In [6]:
response = model.generate_content("What is the capital of Panama? Give short response.")
print(response.text)

The capital of Panama is **Panama City**.


In [7]:
import google.generativeai as genai
print(genai.__version__)

0.8.6


# cleaning up the text

In [8]:
clean_text = str(response.text).replace('*', '')
print(clean_text)

The capital of Panama is Panama City.


# Installing ngrok and setting it up

In [9]:
!pip install -q pyngrok

In [10]:
!pyngrok --version

ngrok version 3.39.10
pyngrok version 8.1.2


In [11]:
with open("ngrok_authenticator_key.txt", "r") as f:
    ngrok_auth_key = f.read()

In [12]:
from pyngrok import ngrok # importing the library
!ngrok config add-authtoken {ngrok_auth_key} # setting the ngrok tunnel to host the website

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


# Creating app.py file that has out chatbot code

In [13]:
user_input = input("say something ")

say something hi!!


In [14]:
user_input

'hi!!'

In [15]:
with open("app.py", "w") as f:
    f.write(f"""# app.py

import streamlit as st
import google.generativeai as genai

# Configure Gemini API
genai.configure(api_key = '{Chat_bot_2}')

# Load model
model = genai.GenerativeModel("gemini-3.5-flash")

# Streamlit page
st.set_page_config(page_title="Gemini Chatbot")

st.title("🤖 Parth's study assistant")

# Input box
user_input = st.text_input("Ask something")

# Generate response
if user_input:
    response = model.generate_content(user_input)
    clean_text = response.text.replace('*', '')
    st.write(clean_text)""")

In [16]:
import streamlit as st
st.set_page_config(page_title="Gemini Chatbot")

2026-07-28 04:47:37.035 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [17]:
with open("ngrok_authenticator_key.txt", "r") as f:
    ngrok_auth_key = f.read()

from pyngrok import ngrok # importing the library
!ngrok config add-authtoken {ngrok_auth_key} # setting the ngrok tunnel to host the website

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [18]:
public_url = ngrok.connect(8501)


# Hosting app on streamlit using ngrok

In [19]:
print(public_url)
!streamlit run app.py &  #to run streamlit app

NgrokTunnel: "https://area-devious-abstract.ngrok-free.dev" -> "http://localhost:8501"


2026-07-28 04:47:47.030 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.243.56:8501

/content/app.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


  Stopping...


# Adding Memory

In [25]:
chat_history = []

In [26]:
chat_history.append(
    "User: My name is Parth"
)

In [27]:
chat_history.append(
    "Assistant: Nice to meet you"
)

In [28]:
chat_history

['User: My name is Parth', 'Assistant: Nice to meet you']

In [29]:
context = "\n".join(chat_history)
print(context)

User: My name is Parth
Assistant: Nice to meet you


In [30]:
type(context)

str

In [31]:
prompt = context + "\nUser: What is my name?"

print(prompt)

User: My name is Parth
Assistant: Nice to meet you
User: What is my name?


In [32]:
response = model.generate_content(prompt)
print(response.text)

Your name is Parth!


In [33]:
SYSTEM_PROMPT = """
You are a friendly AI study assistant.

Rules:
- explain simply
- use examples
- encourage learning
- do not go out of topic
"""

In [34]:
full_prompt = f"""
{SYSTEM_PROMPT}

{context}

Assistant:
"""
print(full_prompt)




You are a friendly AI study assistant.

Rules:
- explain simply
- use examples
- encourage learning
- do not go out of topic


User: My name is Parth
Assistant: Nice to meet you

Assistant:



In [35]:
with open("app1.py", "w") as f:
    f.write(f'''
import streamlit as st
import google.generativeai as genai

gemini_api_key = st.secrets["GEMINI_API_KEY"]

genai.configure(api_key=gemini_api_key)

model = genai.GenerativeModel("gemini-3.5-flash")

SYSTEM_PROMPT = """
You are a helpful AI assistant.
Remember details shared in the conversation and answer accordingly.
"""

st.set_page_config(page_title="Memory Chatbot")

st.title("🧠 Akangsha's Memory Assistant")

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

for message in st.session_state.chat_history:
    with st.chat_message(message["role"]):
        st.write(message["content"])

user_input = st.chat_input("Ask something...")

if user_input:

    st.session_state.chat_history.append(
        {{
            "role": "user",
            "content": user_input
        }}
    )

    conversation_text = ""

    for message in st.session_state.chat_history:

        if message["role"] == "user":
            conversation_text += f"User: {{message['content']}}\\n"

        else:
            conversation_text += f"Assistant: {{message['content']}}\\n"

    full_prompt = f"""
{{SYSTEM_PROMPT}}

{{conversation_text}}

Assistant:
"""

    try:
        response = model.generate_content(full_prompt)
        assistant_reply = response.text

    except Exception as e:
        assistant_reply = f"Error: {{str(e)}}"

    st.session_state.chat_history.append(
        {{
            "role": "assistant",
            "content": assistant_reply
        }}
    )

    st.rerun()

if st.button("🗑️ Clear Chat"):
    st.session_state.chat_history = []
    st.rerun()
''')

In [36]:
for m in genai.list_models():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [ ]:
from pyngrok import ngrok # importing the library
!ngrok config add-authtoken {ngrok_auth_key} # setting the ngrok tunnel to host the website
public_url = ngrok.connect(8501)

print(public_url)
!streamlit run app1.py &  #to run streamlit app

## Knowledge Injection

In [ ]:
with open("notes.txt","r") as f:
    notes = f.read()

print(notes[:500])

In the interplanet tournament, The Mars team dominated the Galactic Sphere League this season, a fast-paced game played with floating energy orbs on anti-gravity arenas. Known for their lightning passes and clever formations, Mars won 18 of 20 matches and captured the Crimson Cup. Captain Akangsha Goswami led the squad with exceptional strategy, while striker Madhuri Dixit scored a league-record 42 orb-goals. Fans packed the Sky Dome every week to watch their thrilling comebacks and spectacular 


In [ ]:
response = model.generate_content("What is the interplanet tournament?")

print(response.text)

Because the term **"Interplanet Tournament"** (or **"Interplanetary Tournament"**) is used in several different fictional universes—primarily in anime, video games, and sci-fi—the exact definition depends on the context you are referring to. 

Here are the most famous "interplanetary tournaments" in pop culture:

---

### 1. *Inazuma Eleven GO: Galaxy* (Grand Celesta Galaxy)
In this popular anime and video game series, the **Grand Celesta Galaxy** is an interplanetary soccer (football) tournament. 
* **The Plot:** Earth is forced to enter a soccer tournament against teams from other planets. If Earth's team (Earth Eleven) loses, the planet will be destroyed by an incoming black hole, or invaded by the alien empires organizing the tournament.
* **Significance:** It is the ultimate test of humanity's "soul" power against diverse alien species with unique abilities.

### 2. *Dragon Ball Z: Bojack Unbound* (The Intergalactic Tournament)
While technically called the "Intergalactic World Tou

In [ ]:
question = "What is interplanet tournament? who won and how?"

prompt = f"""
Use the knowledge below.

KNOWLEDGE:
{notes}

QUESTION:
{question}
"""

In [ ]:
prompt

'\nUse the knowledge below.\n\nKNOWLEDGE:\nIn the interplanet tournament, The Mars team dominated the Galactic Sphere League this season, a fast-paced game played with floating energy orbs on anti-gravity arenas. Known for their lightning passes and clever formations, Mars won 18 of 20 matches and captured the Crimson Cup. Captain Akangsha Goswami led the squad with exceptional strategy, while striker Madhuri Dixit scored a league-record 42 orb-goals. Fans packed the Sky Dome every week to watch their thrilling comebacks and spectacular aerial plays. Despite fierce competition from rival teams Venus and Titan, Mars remained undefeated at home and finished the season as champions.\n\nQUESTION:\nWhat is interplanet tournament? who won and how?\n'

In [ ]:
response = model.generate_content(prompt)

print(response.text)

Based on the provided text:

* **What is the interplanet tournament?** 
It is the Galactic Sphere League, which is described as a fast-paced game played with floating energy orbs on anti-gravity arenas.

* **Who won?** 
The Mars team won the tournament, capturing the Crimson Cup and finishing the season as champions.

* **How did they win?** 
They won by:
* Winning 18 of 20 matches and remaining undefeated at home.
* Employing lightning passes, clever formations, thrilling comebacks, and spectacular aerial plays.
* Being led by Captain Akangsha Goswami's exceptional strategy.
* Having striker Madhuri Dixit score a league-record 42 orb-goals.
